# Required Libraries

In [15]:
import hmac

import numpy as np
import hashlib
import os

from cryptography.hazmat.primitives.asymmetric import x25519
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


# The X3DH stage

In this step, Both parties generate a master key

$$ms = DH_1 || DH_2 || DH_3 || DH_4$$
where

$$DH_1 = DH(IPK_A, SPK_B)$$
$$DH_2 = DH(EK_A, IPK_B)$$
$$DH_3 = DH(EK_A, SPK_B)$$
$$DH_4 = DH(EK_A, OPK_B)$$

then this key goes to a key derivation function and returns both root key and chaining key by the ECDH exponentiation operation between bob's prekey and alice's ratchet key

Generating the keys

In [6]:
# Bob secrets
ik_bob = x25519.X25519PrivateKey.generate()
spk_bob = x25519.X25519PrivateKey.generate()
opk_bob = x25519.X25519PrivateKey.generate()

# bob's bundle

bundle_bob = {
    'ik': ik_bob.public_key(),
    'spk': spk_bob.public_key(),
    'opk': opk_bob.public_key(),
}

# Alice secrets

ik_alice = x25519.X25519PrivateKey.generate()
ek_alice = x25519.X25519PrivateKey.generate()

ik <cryptography.hazmat.bindings._rust.openssl.x25519.X25519PublicKey object at 0x7fbc42b3a1d0>
spk <cryptography.hazmat.bindings._rust.openssl.x25519.X25519PublicKey object at 0x7fbbf9eff9f0>
opk <cryptography.hazmat.bindings._rust.openssl.x25519.X25519PublicKey object at 0x7fbbf9eff7d0>


Key Derivation Function

In [7]:
def derive_keys(master_secret):
    return HKDF(
        algorithm=hashes.SHA256(),
        length=64,
        salt=b'\x00' * 32,
        info=b"Signal_X3DH_Handshake",
    ).derive(master_secret)

Computing the master secret

In [8]:
dh1 = ik_alice.exchange(bundle_bob['spk'])
dh2 = ek_alice.exchange(bundle_bob['ik'])
dh3 = ek_alice.exchange(bundle_bob['spk'])
dh4 = ek_alice.exchange(bundle_bob['opk'])

alice_ms = dh1 + dh2 + dh3 + dh4


Deriving alice's root keys

In [9]:
alice_keys = derive_keys(alice_ms)
alice_root_key, alice_chain_key = alice_keys[:32], alice_keys[32:]

Computing Bob's master secret

In [11]:
bob_dh1 = spk_bob.exchange(ik_alice.public_key())
bob_dh2 = ik_bob.exchange(ek_alice.public_key())
bob_dh3 = spk_bob.exchange(ek_alice.public_key())
bob_dh4 = opk_bob.exchange(ek_alice.public_key())

bob_ms = bob_dh1 + bob_dh2 + bob_dh3 + bob_dh4

bob_keys = derive_keys(bob_ms)
bob_root_key, bob_chain_key = bob_keys[:32], bob_keys[32:]

Checking

In [12]:
print(f"Alice's root key: {alice_root_key.hex()}")
print(f"Bob's root key: {bob_root_key.hex()}")
assert alice_root_key == bob_root_key

Alice's root key: d433ad4180fc3cad62faafe08a501567b8362422c51e28b71a1040caaa93e298
Bob's root key: d433ad4180fc3cad62faafe08a501567b8362422c51e28b71a1040caaa93e298


# Symmetric Ratcheting

To derive from chaining key a Message key

In [13]:
def kdf_ratchet(chain_key):
    msg_key = hmac.new(chain_key, b"\x01", hashlib.sha256).digest()
    next_chain = hmac.new(chain_key, b"\x02", hashlib.sha256).digest()
    return msg_key, next_chain

In [17]:
alice_chain_key, msg_key_1 = kdf_ratchet(alice_chain_key)
alice_chain_key, msg_key_2 = kdf_ratchet(alice_chain_key)

print(f"Msg 1 Key: {msg_key_1.hex()[:16]}...")
print(f"Msg 2 Key: {msg_key_2.hex()[:16]}...")

Msg 1 Key: 07d5405016ebb07c...
Msg 2 Key: 7c5b2e9197bd6c2f...


# Asymmetric Ratcheting

Refreshes the entropy and provides future secrecy

In [18]:
def dh_ratchet_update(root_key, alice_private, bob_public_keys):
    shared_secret = alice_private.exchange(bob_public_keys)

    res = hmac.new(root_key, shared_secret, hashlib.sha256).digest()
    return res[:16], res[16:]

In [21]:
bob_new_ephemeral = x25519.X25519PublicKey.from_public_bytes(os.urandom(32))
alice_ratchet_priv = x25519.X25519PrivateKey.generate()

alice_root_key, chain_key_sent = dh_ratchet_update(alice_root_key, alice_ratchet_priv, bob_new_ephemeral)

print(f"The new root key: {alice_root_key.hex()}")

The new root key: 4dd1523efa2ec4f91ce3124902b8a6e9


# Potential attacks

The Key Compromise Impersonation

In [25]:
ik_alice_priv = x25519.X25519PrivateKey.generate()
ik_bob = x25519.X25519PrivateKey.generate() # Bob's real public key
spk_bob = x25519.X25519PrivateKey.generate()
opk_bob = x25519.X25519PrivateKey.generate()

bundle_bob = {
    'ik': ik_bob.public_key(),
    'spk': spk_bob.public_key(),
    'opk': opk_bob.public_key(),
}

In [24]:
eve_ik_priv = ik_alice_priv
eve_ephemeral_priv = x25519.X25519PrivateKey.generate()

In [30]:
dh1 = eve_ik_priv.exchange(bundle_bob['spk'])
dh2 = eve_ephemeral_priv.exchange(bundle_bob['ik'])
dh3 = eve_ephemeral_priv.exchange(bundle_bob['spk'])
dh4 = eve_ephemeral_priv.exchange(bundle_bob['opk'])

eve_ms = dh1 + dh2 + dh3 + dh4

eve_keys = derive_keys(eve_ms)
eve_root_key, eve_chain_key = eve_keys[:32], eve_keys[32:]

bob_dh1 = spk_bob.exchange(eve_ik_priv.public_key())
bob_dh2 = ik_bob.exchange(eve_ephemeral_priv.public_key())
bob_dh3 = spk_bob.exchange(eve_ephemeral_priv.public_key())
bob_dh4 = opk_bob.exchange(eve_ephemeral_priv.public_key())

bob_ms = bob_dh1 + bob_dh2 + bob_dh3 + bob_dh4

bob_keys = derive_keys(bob_ms)
bob_root_key, bob_chain_key = bob_keys[:32], bob_keys[32:]

print(f"Eve's root key: {eve_root_key.hex()}")
print(f"Bob's root key: {bob_root_key.hex()}")
assert eve_root_key == bob_root_key

Eve's root key: 6f090e8fd7349b411c051955b6840dcaec4534926eb1af21226b0149ec31bf3f
Bob's root key: 6f090e8fd7349b411c051955b6840dcaec4534926eb1af21226b0149ec31bf3f


# Unknown Key Share

In [31]:
alice_identity = b"Alice"
bob_identity = b"Bob"
eve_identity = b"Eve"

shared_secret = b"some_dh_output"

key_no_binding = hashlib.sha256(shared_secret).digest()
key_binding = hashlib.sha256(shared_secret + alice_identity + bob_identity).digest()

print(f"Standard Key: {key_no_binding.hex()[:16]}")
print(f"Mitigated Key (Bound): {key_binding.hex()[:16]}")

Standard Key: c04874997e6c44e7
Mitigated Key (Bound): 80f57d432e8f3fe4


# State Shapshot Attack

In [32]:
captured_ck = b"some_stolen_key_1"

ck2, mk2 = kdf_ratchet(captured_ck)
ck3, mk3 = kdf_ratchet(ck2)

eve_ck = captured_ck
eve_next_ck, eve_mk2 = kdf_ratchet(eve_ck)
eve_next_ck, eve_mk3 = kdf_ratchet(eve_next_ck)

print(f"Alice's MK2: {mk2.hex()[:10]}")
print(f"Eve's Stolen MK2: {eve_mk2.hex()[:10]}")
assert mk2 == eve_mk2, "Eve successfully decrypted future messages in this chain!"

Alice's MK2: 20d0a87d20
Eve's Stolen MK2: 20d0a87d20


# Proposed Implementation of Signal Protocol: ML-LWE/SIS + STS

**Configuration parameters**

In [33]:
n = 256  #Polynomial Degree
k = 3    #Module-LEW/SIS rank
q = 7681 #The modulus
eta = 4  #Binomial distribution param
dt = 11  #Compression bits for Alice
du = 11  #Compression bits for Bob

In [34]:
roh = os.urandom(32)

np.random.seed(int.from_bytes(roh, 'big') % 2 ** 32)
A = np.random.randint(0, q, (k, k, n))

In [35]:
def cbd(eta, size):
    x = np.random.randint(0, 2, (eta, *size))
    y = np.random.randint(0, 2, (eta, *size))
    return np.sum(x - y, axis=0)

In [36]:
s_A = cbd(eta, (k, n))
e_A = cbd(eta, (k, n))